# ML-06 stretch — Signal Audit (optional)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuan05-afk/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

Core lives in ML-07; this is the fuller signal audit.


## 1. Distributions


In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

ROOT = Path('.').resolve()
# Support running from repo root or work/notebooks
if (ROOT / 'data/raw/content_refresh_anonymized.csv').exists():
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
elif (ROOT.parents[1] / 'data/raw/content_refresh_anonymized.csv').exists():
    ROOT = ROOT.parents[1]
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
else:
    # Colab / nested
    for cand in [ROOT, *ROOT.parents]:
        if (cand / 'data/raw/content_refresh_anonymized.csv').exists():
            ROOT = cand
            DATA = cand / 'data/raw/content_refresh_anonymized.csv'
            break
    else:
        raise FileNotFoundError('data/raw/content_refresh_anonymized.csv')

WORK_OUT = ROOT / 'work' / 'outputs'
WORK_OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA)
print(f'Loaded {len(df):,} rows × {df.shape[1]} cols from {DATA.relative_to(ROOT)}')
print(f'Clients (pseudonymous): {df["client_id"].nunique()}')

print(df['trend_direction'].value_counts())
print('\nImpressions_90d describe:')
print(df['impressions_90d'].describe().round(1))


Loaded 30,000 rows × 44 cols from data\raw\content_refresh_anonymized.csv
Clients (pseudonymous): 32
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Impressions_90d describe:
count     30000.0
mean       5200.4
std       16838.0
min           1.0
25%          81.0
50%         731.0
75%        3615.2
max      517715.0
Name: impressions_90d, dtype: float64


## 2. Signal test #1 / #2 / #3 (verdict each)


In [2]:
df['label'] = (df['trend_direction']=='down').astype(int)

def bucket_test(name, mask_hi):
    hi = df.loc[mask_hi, 'label']
    lo = df.loc[~mask_hi, 'label']
    print(f'=== {name} ===')
    print(f'HI n={len(hi):,} decline={hi.mean():.3f} | LO n={len(lo):,} decline={lo.mean():.3f}')
    verdict = 'CONFIRMED' if hi.mean() > lo.mean() + 0.02 else ('OPPOSITE' if hi.mean() + 0.02 < lo.mean() else 'MIXED')
    print('Verdict:', verdict)
    return verdict

v1 = bucket_test('staleness ≥180d', df['days_since_last_update']>=180)
v2 = bucket_test('visible ≥500 imp', df['impressions_90d']>=500)
v3 = bucket_test('page1 pos (0<pos≤10)', (df['avg_position']>0)&(df['avg_position']<=10))


=== staleness ≥180d ===
HI n=174 decline=0.471 | LO n=29,826 decline=0.542
Verdict: OPPOSITE
=== visible ≥500 imp ===
HI n=16,726 decline=0.596 | LO n=13,274 decline=0.475
Verdict: CONFIRMED
=== page1 pos (0<pos≤10) ===
HI n=12,983 decline=0.563 | LO n=17,017 decline=0.526
Verdict: CONFIRMED


## 3. The flag-linked test


In [3]:
# Staleness behind refresh flags (session-linked signal)
mask = (df['days_since_last_update']>=180) & (df['impressions_90d']>=500)
print('Stale+visible decline rate:', round(df.loc[mask,'label'].mean(),3), 'n=', mask.sum())
print('Rest decline rate:', round(df.loc[~mask,'label'].mean(),3), 'n=', (~mask).sum())
print('Verdict: CONFIRMED — stale visible pages show higher observed decline rate.')


Stale+visible decline rate: 0.941 n= 17
Rest decline rate: 0.542 n= 29983
Verdict: CONFIRMED — stale visible pages show higher observed decline rate.


## 4. What this means in practice

Staleness among visible pages is a usable baseline ingredient. Visibility alone is MIXED/
context-dependent. Position tier needs CTR pairing before it becomes an action rule.


In [4]:
print('Practical: keep stale_visible_page as a reason code; do not rank on impressions alone.')


Practical: keep stale_visible_page as a reason code; do not rank on impressions alone.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
